In [ ]:
import os
import time
import urllib.request

import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import pyautogui


# -------------------------------------------------------
# 1. Скачать модель, если её нет
# -------------------------------------------------------


MODEL_PATH = "hand_landmarker.task"


# -------------------------------------------------------
# 2. Функция отрисовки рук
# -------------------------------------------------------

HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (17, 18), (18, 19), (19, 20),
    (0, 17)
]


def draw_hand_landmarks(frame, detection_result):
    height, width, _ = frame.shape

    hands = detection_result.hand_landmarks
    handedness_list = getattr(detection_result, "handedness", [])

    for hand_index, landmarks in enumerate(hands):
        # Рисуем точки
        for landmark in landmarks:
            x = int(landmark.x * width)
            y = int(landmark.y * height)
            cv2.circle(frame, (x, y), 3, (0, 255, 0), -1)

        # Рисуем соединения
        for connection in HAND_CONNECTIONS:
            start_index, end_index = connection

            start = landmarks[start_index]
            end = landmarks[end_index]

            x1 = int(start.x * width)
            y1 = int(start.y * height)

            x2 = int(end.x * width)
            y2 = int(end.y * height)

            cv2.line(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)

        # Подпись Left / Right, если доступна
        if handedness_list and hand_index < len(handedness_list):
            categories = handedness_list[hand_index]

            if categories:
                category = categories[0]

                label = getattr(category, "category_name", "")
                if not label:
                    label = getattr(category, "display_name", "")

                score = getattr(category, "score", 0.0)

                wrist = landmarks[0]
                x = int(wrist.x * width)
                y = int(wrist.y * height)

                text = f"{label} {score:.2f}"

                cv2.putText(
                    frame,
                    text,
                    (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 0, 255),
                    2
                )

    return frame


# -------------------------------------------------------
# 3. Камера
# -------------------------------------------------------

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Не открылась камера")
    exit()

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)


# -------------------------------------------------------
# 4. MediaPipe HandLandmarker
# -------------------------------------------------------

BaseOptions = python.BaseOptions
HandLandmarker = vision.HandLandmarker
HandLandmarkerOptions = vision.HandLandmarkerOptions
RunningMode = vision.RunningMode

base_options = BaseOptions(model_asset_path=MODEL_PATH)

options = HandLandmarkerOptions(
    base_options=base_options,
    running_mode=RunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5
)


# -------------------------------------------------------
# 5. Основной цикл
# -------------------------------------------------------

timestamp_ms = 0

screen_width, screen_height = pyautogui.size()

with HandLandmarker.create_from_options(options) as detector:
    while True:
        ret, frame = cap.read()

        if not ret:
            print("Не удалось получить кадр")
            break

        # Зеркалим изображение
        frame = cv2.flip(frame, 1)

        # MediaPipe Tasks требует RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Создаём MediaPipe Image
        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )

        # Для VIDEO mode нужно передавать растущий timestamp
        timestamp_ms += 33

        detection_result = detector.detect_for_video(
            mp_image,
            timestamp_ms
        )

        hands = detection_result.hand_landmarks
        
        if hands:
            
            coordinates = hands[0][8]
            
            x = coordinates.x * screen_width
            y = coordinates.y * screen_height

            print(x, y)

            pyautogui.moveTo(x, y, duration=0)

        # Рисуем результат
        annotated_image = draw_hand_landmarks(frame, detection_result)

        cv2.imshow("MediaPipe HandLandmarker", annotated_image)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break


cap.release()
cv2.destroyAllWindows()

1832.5366973876953 795.6861019134521
1589.8504257202148 1217.5761222839355
1499.6126747131348 1130.4244995117188
1540.9894180297852 1162.4692797660828
1463.4468841552734 1074.5129942893982
1524.3720817565918 1128.9266467094421
1454.234848022461 1089.6073937416077
1525.5555152893066 1135.9118700027466
1462.880916595459 1075.7651567459106
1468.2515144348145 1065.6584858894348
1489.6082496643066 1089.608359336853
1502.7159690856934 1101.8962025642395
1496.147518157959 880.0176501274109
1592.0823669433594 674.5701491832733
1894.0108680725098 1070.0653553009033
589.2426538467407 817.5054967403412
687.9852390289307 536.180716753006
977.9291152954102 707.3734045028687
1374.025468826294 1412.8632545471191
1752.173080444336 888.0399227142334
1875.9431648254395 952.2953510284424
1758.4035301208496 1021.1111783981323
1541.5970993041992 907.4730634689331
1389.618844985962 869.8589444160461
1336.907558441162 832.2793185710907
1490.9777641296387 774.0650832653046
1490.7740020751953 752.7619063854218

FailSafeException: PyAutoGUI fail-safe triggered from mouse moving to a corner of the screen. To disable this fail-safe, set pyautogui.FAILSAFE to False. DISABLING FAIL-SAFE IS NOT RECOMMENDED.

: 